# Workshop: Dictionaries and Tuples

### February 7, 2024

In [5]:
# Review: memoization pattern to compute Fibonacci numbers.

fibo_lookup = dict()

def fibo( n ): # Fn = F(n-1)+F(n-2)
    if n in fibo_lookup:
        return fibo_lookup[n]
    if n==0:
        fibo_lookup[n]=0
        return 0
    if n==1:
        fibo_lookup[n]=1
        return 1
    fn = fibo( n-1 ) + fibo( n-2 ) 
    fibo_lookup[n] = fn
    return fn

In [7]:
fibo( 0 ), fibo(1), fibo(2), fibo(3), fibo(100)

(0, 1, 1, 2, 354224848179261915075)

## Problem 1: Memoization revisited

Adapted from Downey Exercise 10.2 (https://greenteapress.com/thinkpython2/html/thinkpython2012.html#sec139)

The Ackermann function, $A(m, n)$, is defined as

$$ A(m, n) = \begin{cases}
                n+1	& \mbox{ if }  m = 0 \\
                A(m−1, 1) &\mbox{ if }  m > 0 \text{ and } n = 0 \\
                A(m−1, A(m, n−1)) &\mbox{ if }  m > 0 \text{ and } n > 0.
                \end{cases}
                $$
See https://en.wikipedia.org/wiki/Ackermann_function for more information.

Write a function named `ack` that takes two non-negative integers, `m` and `n` and returns $A(m,n)$.
Include error checking to verify that both arguments are non-negative integers, and raise an appropriate error if not.

Use your function to evaluate `ack(3, 4)`, which should be 125.

What happens for larger values of `m` and `n`, say, `ack(4,4)`?

In [10]:
def ack( m, n):
    if not isinstance( m, int ):
        raise TypeError('m should be an integer.')
    if m < 0:
        raise ValueError('m should be non-negative.')
    if not isinstance( n, int ):
        raise TypeError('n should be an integer.')
    if n < 0:
        raise ValueError('n should be non-negative.')

    if m==0:
        return n+1
    # If m !=0, it must be >0.
    if n==0:
        return ack( m-1, 1)
    else: # n > 0
        return ack( m-1, ack(m,n-1) )

In [11]:
ack(3,4) # Should return 125

125

In [12]:
ack(4,4) # SHould return... ?? Try running it

RecursionError: maximum recursion depth exceeded

Now, try memoizing this function, similar to how we memoized the Fibonacci sequence in lecture. Call your new function `A_memo`, which should take the same arguments as `A`.

Does your memoized function get any faster?

In [18]:
known = dict()

def A_memo( m, n ):
    if not isinstance( m, int ):
        raise TypeError('m should be an integer.')
    if m < 0:
        raise ValueError('m should be non-negative.')
    if not isinstance( n, int ):
        raise TypeError('n should be an integer.')
    if n < 0:
        raise ValueError('n should be non-negative.')
    
    if (m,n) in known:
        return known[ (m,n) ]
    if m==0:
        a = n+1
        known[ (m,n) ] = a
        #return A_memo( m,n ) # Any of these returns would work.
        #return known[ (m,n) ] 
        return a
    # If m !=0, it must be >0.
    if n==0:
        a = ack(m-1,1)
        known[ (m,n) ] = a
        return a
    else: # n > 0
        second_arg = ack(m,n-1)
        a = ack(m-1, second_arg)
        known[ (m,n) ] = a
        return a

In [19]:
A_memo(3,4) # Should still return 125.

125

In [20]:
A_memo(4,4) # If you try running this, you'll see that we still have a recursion error. What's up with that?

RecursionError: maximum recursion depth exceeded

Well, the problem arises when we recurse with `A_memo(m-1, A_memo(m,n-1))`.

Python first tries to evaluate `A_memo(m,n-1)`. Unless we're lucky (i.e., have computed `A(m,n-1)`), this in turn results in a call to `A_memo(m,n-2)`, recursing until we have a known value or, as in the case of our computation above, we reach `A_memo(m,0)`, and try to compute `A_memo(m-1,1)`.

So, in the case of $A(4,4)$, if we already know $A(3,4)$, this should look like
$$
A(4,4) \rightarrow A(4,3) \rightarrow A(4,2) \rightarrow A(4,1) \rightarrow A(4,0)
\rightarrow A(4,0) \rightarrow A(3,1),
$$
and if we already know $A(3,4)$, we know $A(3,1)$, so $(3,1)$ will be memoized.
That doesn't look bad at all. What's going wrong?

Well, perhaps the problem is this: `A_memo(3, A_memo(4,3))` first evaluates `A_memo(4,3)`, finds that the answer is whatever it is, say `a43`, and tries to evaluate `A_memo(3, a43)`, resulting in a whole new mess of recursion!
If we're not careful, memoization doesn't actually help us!


Indeed, you can prove that no matter what you do, memoization can't help very much here.
If you work out the call graph of the function, there aren't a whole lot of repetitions in the way that there are with the Fibonacci computations.

The lesson is that memoization only helps us when there are a <i>large number</i> of <i>repeated computations</i> that we are trying to avoid.
If the problem doesn't actually involve repeatedly computing the same thing again and again, memoization doesn't save much time!


## Problem 2: Duplicating duplicates

Adapted from Downey Exercise 

In our previous workshop, we wrote a function `has_duplicates`, which takes a single list as its only argument, and returns a Boolean encoding whether or not the list contains duplicate elements.
Use a dictionary to create this faster version of this function.

In [22]:
# range keyword lets us iterate over indices of a list of a given length
for i in range(10):
    print(i)

0
1
2
3
4
5
6
7
8
9


In [34]:
def duplicates( t ): # Naive version which looks through the list repeatedly.
    for i in range(len(t)):
        e = t[i]
        # This would also work instead of using "in" keyword, but probably a bit slower.
        #for x in t[(i+1):]:
        #    if e==x:
        #        return True
        if e in t[(i+1):]:
            return True # Found a duplicate
    # Looked through whole list, found no dups
    return False 

def duplicates_fast( t ):
    # TODO: error checking

    d = dict() # Tracks things we've seen before.
    
    # For each element of the list, check if it is already in the dictionary
    for e in t:
        # If it is, then we've found a duplicate. Return True.
        if e in d:
            return True
        # Otherwise, we haven't seen this element before. 
        else:
            # Add it to the dictionary and proceed to next element.
            d[e] = 'frog'
    # If we get to the end of the list without finding a duplicate, return False
    return False

In [35]:
duplicates( [1,2,3,4] ) # Should return False

False

In [36]:
duplicates( [1,2,2,4] ) # Should return True

True

In [37]:
duplicates_fast( [1,2,3,4] ) # Should return False

False

In [38]:
duplicates_fast( [1,2,2,4] ) # Should return True

True

Let's use the `time` module to compare our newer, faster program against our old slow one. To do this, we need a long list to run on, and we need lists both with and without duplicates.

Toward that end, write two functions:

- `create_nodup_list( n )`: `n` is a non-negative integer. This function should create a list of the integers `0` through `n-1` and then randomize the order of that list. That is, a list of length `n` with no duplicates. Recall that you can do this using `range(n)`.
- `create_duped_list( n )`: `n` is a non-negative integer. This function should create a list of the integers `0` through `n-1`, but then choose a random number between 0 and `n-1` inclusive, append that random number to the end of the list, and randomly shuffle the list. Thus, this should return a list of length `n+1` that includes a single duplicated entry.

<b>Hint:</b> the Python module `random` includes a function `shuffle` such that if `t` is a list, `random.shuffle(t)` randomly orders the elements of `t` in place. Note that this function doesn't return anything-- it modifies the list `t`. We'll have lots to say about this in a couple of weeks when we talk about functional programming.

<b>Hint:</b> `random.randrange(n)` will produce a random integer from $\{0,1,2,...,n-1\}$.'

In [39]:
# Example of using the random.shuffle
import random
t = list( range(20) ) # random.shuffle won't take a range object-- make it a list
print( t )
random.shuffle( t)
t

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


[15, 10, 9, 19, 7, 8, 1, 4, 5, 13, 14, 17, 11, 0, 2, 12, 16, 3, 6, 18]

In [40]:
random.randrange(10) # Pick a random number between 0 and 10.

1

In [41]:
def create_nodup_list( n ):
    t = list( range( n ) )
    random.shuffle( t) # Shuffle t in place.
    return t

In [42]:
create_nodup_list( 10 )

[6, 0, 1, 7, 5, 4, 8, 2, 9, 3]

In [47]:
def create_duped_list( n ):
    t = list( range( n ) )
    random.shuffle(t) # Shuffle t in place.
    # Randomly choose a number between 0 and n-1 inclusive.
    e = random.randrange(n)
    # Append it to the end of the list.
    # Can use t = t +[e]
    t.append(e) # https://docs.python.org/3/tutorial/datastructures.html#more-on-lists
    return t

In [48]:
create_duped_list( 10 )

[0, 1, 6, 5, 8, 9, 4, 7, 3, 2, 4]

Okay, time for the payoff. Use the `time` module to compare our two different implementations of duplicate checking! Which one is better? Try a variety of list lengths-- depending on how fast your computer is, you should start to see a measurable difference once the length is between a few hundred and a few thousand.
You may find it useful to repeat the experiment several times and take the average-- `sum(t)/len(t)` computes the mean of a list `t`, provided adding elements of `t` is permitted.

In [53]:
import time
n = 10000 # Number of list elements
# Create two lists.
list_dup = create_duped_list( n )
list_nodup = create_nodup_list( n )

# Time the list-based (slow duplicates)
t1 = time.time()
duplicates( list_dup )
duplicates( list_nodup )
t2 = time.time()
slow_time = t2-t1

# Time the dict-based (fast duplicates)
t1 = time.time()
duplicates_fast( list_dup )
duplicates_fast( list_nodup )
t2 = time.time()
fast_time = t2-t1

In [54]:
slow_time, fast_time

(1.481698989868164, 0.0023097991943359375)

## Problem 3: interleaving

Write a function called `interleaf(t1,t2)`, where `t1` and `t2` are tuples, and returns a new tuple that is obtained by "interleaving" the tuples: we take the first element of `t1` and make it the first element of our new tuple.
We take the first element of `t2` and make it the second element of our new tuple.
The second element of `t1` becomes the third element of our new tuple, and so on.
If we run out of elements from one or the other of tuples `t1` and `t2`, simply append what is left of the other tuple onto the result.
Think carefully about what should happen if one or the other tuple is empty.
<b>Hint:</b> there is a particularly simple/elegant solution to this problem using recursion.

For example, `interleaf( (1,2,3,4), ('a','b') )` should return `(1,'a',2,'b',3,4)`.

Your function should check that both `t1` and `t2` are tuples, and raise an appropriate error if not.

In [ ]:
def interleaf( t1, t2 ):
    if not isinstance(t, (str, list, tuple)):
        # Really, we could extend this list of types further,
        # but we'll leave that matter for a few weeks from now
        # when we talk about iterators.
        raise TypeError('t should be sequence data')
    
    # If either of the tuples are empty, we are done.
    if len(t1)==0:
        return t2
    elif len(t2)==0:
        return t1
    else: # Recurse. Take the first element of each one, glom them together, and recurse.
        return t1[:1] + t2[:1] + interleaf(t1[1:], t2[1:])

In [ ]:
interleaf( [1,2,3], ['a','b','c']) # Should return [1, 'a', 2, 'b', 3, 'c']

In [ ]:
interleaf( (1,2,3,4), ('a','b') )